In [2]:
import os
import pickle
import mne
import numpy as np
import matplotlib.pyplot as plt

from itertools import combinations
from scipy.stats import ttest_ind
from mne.stats import f_mway_rm
from mne.stats import f_threshold_mway_rm
from util_regions import *

# GLOBAL VARS
ROOT = '/Volumes/Server/NEUROLING/PersonalFiles/Nigel Flower/brainquest'
os.chdir(ROOT)

mri_dir = os.path.join(ROOT, 'mri')
stc_dir = os.path.join(ROOT, 'stc')
stats_dir = os.path.join(ROOT, 'stats')

# Configure experiment variables

In [3]:
exp_dirs = ['structure_signed', 'transposed_signed']
exp_dirs = ['structure_signed', 'transpose_signed']

experiments = ['exp1', 'exp2', 'both']
comparisons = ['sentences', 'anova', 'nouns']

exp_dir_dict = {
    experiments[0]: exp_dirs[0],
    experiments[1]: exp_dirs[1]
}

In [10]:
experiment = experiments[1]
comparison = comparisons[0]

# Specify Subject List

## Experiment 1

["R1260","R1421","R1600","R1718","R1840",
            "R1891","R1898","R1905","R1907","R1908",
            "R1914","R1917","R1921","R1928","R1934",
            "R1942","R1988","R2076","R2077","R2079",
            "R2089"]
## Experiment 2
["R1260","R1421","R1600","R1718","R1840",
            "R1890","R1891","R1897","R1898","R1904",
            "R1905","R1907","R1908","R1914","R1917",
            "R1921","R1928","R1934","R1942","R1988",
            "R2076","R2077","R2079","R2089"]

In [5]:
exp1_subjects = ["R1260","R1421","R1600","R1718","R1840",
                 "R1891","R1898","R1905","R1907","R1908",
                 "R1914","R1917","R1921","R1928","R1934",
                 "R1942","R1988","R2076","R2077","R2079",
                 "R2089"]

exp2_subjects = ["R1260","R1421","R1600","R1718","R1840",
                 "R1890","R1891","R1897","R1898","R1904",
                 "R1905","R1907","R1908","R1914","R1917",
                 "R1921","R1928","R1934","R1942","R1988",
                 "R2076","R2077","R2079","R2089"]

combined_subjects = sorted(list(set(exp1_subjects) & set(exp2_subjects)))

In [6]:
exp1_conditions = ['all_bare',  'all_noun',  'all_sent', 
                   'some_bare', 'some_noun', 'some_sent',
                   'no_bare',   'no_noun',   'no_sent',
                   'the_bare',  'the_noun',  'the_sent',
                   'one_noun',  'two_noun',  'three_noun']

exp2_conditions = ['all_sent',  'all_trans1',  'all_trans2', 
                   'some_sent', 'some_trans1', 'some_trans2',
                   'no_sent',   'no_trans1',   'no_trans2',
                   'the_sent',  'the_trans1',  'the_trans2']

noun_conditions = ['all_noun', 'some_noun', 'no_noun', 'the_noun']

sentence_conditions = ['all_sent', 'some_sent', 'no_sent', 'the_sent']

sentence_conditions_comb = ['all_sent_exp1', 'some_sent_exp1', 'no_sent_exp1', 'the_sent_exp1',
                            'all_sent_exp2', 'some_sent_exp2', 'no_sent_exp2', 'the_sent_exp2']

In [11]:
subjects = []
conditions = []

if experiment == 'exp1' and comparison == 'anova':    
    subjects = exp1_subjects
    conditions = exp1_conditions
elif experiment == 'exp1' and comparison == 'sentences':
    subjects = exp1_subjects
    conditions = sentence_conditions
elif experiment == 'exp1' and comparison == 'nouns':
    subjects = exp1_subjects
    conditions = noun_conditions
elif experiment == 'exp2' and comparison == 'anova':
    subjects = exp2_subjects
    conditions = exp2_conditions
elif experiment == 'exp2' and comparison == 'sentences':
    subjects = exp2_subjects
    conditions = sentence_conditions
elif experiment == 'both':
    subjects = combined_subjects
    conditions = sentence_conditions_comb
else:
    pass

# Read in the STCs

In [12]:
# data structure
epoch_tmin = -0.1
epoch_tmax = 0.8
times = np.arange(int(epoch_tmin*1000), int(epoch_tmax*1000+1), 1)
n_times = len(times)        # number of time points
n_cond = len(conditions)    # number of conditions
n_subj = len(subjects)      # number of subjects
n_hemisources = 2562        # number of sources in one hemisphere
stcs = []                   # empty list for appending stc files

n_sources = n_hemisources

def get_STC(exp_dir, subj, exp, cond):
    
    stc_fname = os.path.join(stc_dir, exp_dir, 
                                 '%s','%s_%s_%s_dSPM') % (cond, subj, exp, cond)
    
    stc = mne.read_source_estimate(stc_fname, subject='fsaverage')
    
    return stc

for c, cond in enumerate(conditions):
    print('Reading in STCs for condition %s' % cond) 
    
    for s, subj in enumerate(subjects):
        
        exp_dir = None
        subj_exp = subj + '_' + experiment
        stc = None
        
        if experiment == 'both':
            curr_exp = cond.split('_')[-1]
            exp_dir = exp_dir_dict[curr_exp]
            cond_name = '_'.join(cond.split('_')[:-1])
            stc = get_STC(exp_dir, subj, curr_exp, cond_name)
            
        else:
            exp_dir = exp_dir_dict[experiment]
            stc = get_STC(exp_dir, subj, experiment, cond)
        
        stcs.append(stc)

Reading in STCs for condition all_sent
Reading in STCs for condition some_sent
Reading in STCs for condition no_sent
Reading in STCs for condition the_sent


In [22]:
# The variable storing all of the data will be in data
data = []

for c, cond in enumerate(conditions):

    data_tmp = np.empty((n_subj, n_sources, n_times))

    for s, subj in enumerate(subjects):
        
        # get the STC data from Subject s for Condition c
        idx = s+c*n_subj
        subj_stc = stcs[idx]
        
        # Grab all of the left hemisphere sources from the STC object
        # transpose the matrix into: subjects x times x sources
        data_tmp[s,:,:] = subj_stc.data[:n_hemisources]

    data.append(np.transpose(data_tmp, [0, 2, 1]))

In [25]:
data[0].shape

(24, 901, 2562)

# Define ROI Labels

## Aparc Parcellation

In the following cells, I am using brain regions using the Aparc parcellation, and I am mostly using the ROIs defined in Tulling et al. (2021), since Maxime's study involves the comprehension of modal verbs, which in some sense are involved in quantification (over possible worlds). 

In [ ]:
mne.datasets.fetch_aparc_sub_parcellation(subjects_dir=subjects_dir,verbose=True)

annot_name = 'aparc'
left_labels = mne.read_labels_from_annot('fsaverage', annot_name, subjects_dir = subjects_dir, hemi = 'lh')
right_labels = mne.read_labels_from_annot('fsaverage', annot_name, subjects_dir = subjects_dir, hemi = 'rh')

TP = 'temporalpole'
IPS = 'superiorparietal'
supramarginal = 'supramarginal'
inferior_parietal = 'inferiorparietal'
PCC = 'posteriorcingulate'
precuneus = 'precuneus'
rrACC = 'rostralanteriorcingulate'
vmPFC = 'medialorbitofrontal'

In [ ]:
hemi = 'lh'

left_TP        = [label for label in left_labels if label.name == '%s-%s' % (TP, hemi)][0]
left_IPS       = [label for label in left_labels if label.name == '%s-%s' % (IPS, hemi)][0]
left_TPJ       = [label for label in left_labels if label.name == '%s-%s' % (supramarginal, hemi)][0] + \
                 [label for label in left_labels if label.name == '%s-%s' % (inferior_parietal, hemi)][0]
left_PCC       = [label for label in left_labels if label.name == '%s-%s' % (PCC, hemi)][0]
left_Precuneus = [label for label in left_labels if label.name == '%s-%s' % (precuneus, hemi)][0]
left_rrACC     = [label for label in left_labels if label.name == '%s-%s' % (rrACC, hemi)][0]
left_vmPFC     = [label for label in left_labels if label.name == '%s-%s' % (vmPFC, hemi)][0]

In [ ]:
hemi = 'rh'

right_TP        = [label for label in right_labels if label.name == '%s-%s' % (TP, hemi)][0]
right_IPS       = [label for label in right_labels if label.name == '%s-%s' % (IPS, hemi)][0]
right_TPJ       = [label for label in right_labels if label.name == '%s-%s' % (supramarginal, hemi)][0] + \
                  [label for label in right_labels if label.name == '%s-%s' % (inferior_parietal, hemi)][0]
right_PCC       = [label for label in right_labels if label.name == '%s-%s' % (PCC, hemi)][0]
right_Precuneus = [label for label in right_labels if label.name == '%s-%s' % (precuneus, hemi)][0]
right_rrACC     = [label for label in right_labels if label.name == '%s-%s' % (rrACC, hemi)][0]
right_vmPFC     = [label for label in right_labels if label.name == '%s-%s' % (vmPFC, hemi)][0]

## Set up ROI for spatiotemporal clustering

In [26]:
# included area
labels   = mne.read_labels_from_annot('fsaverage', 'PALS_B12_Lobes', 'both', subjects_dir=mri_dir)
roi_name = 'LOBE.TEMPORAL'
roi_lh   = [label for label in labels if label.name == f'{roi_name}-lh'][0]
roi_rh   = [label for label in labels if label.name == f'{roi_name}-rh'][0]

# look for vertex indices in the ROIs 
hemi_idx    = np.arange(0, n_hemisources, 1)
roi_lh_idx  = roi_lh.get_vertices_used(vertices=hemi_idx)
#roi_rh_idx  = roi_rh.get_vertices_used(vertices=hemi_idx)
#roi_bh_idx  = np.concatenate((roi_lh_idx, roi_rh_idx+n_hemisources), axis=0)

# look for vertex indices NOT in the ROIs 
lh_diff_idx = np.setdiff1d(hemi_idx, roi_lh_idx)
#rh_diff_idx = np.setdiff1d(hemi_idx, roi_rh_idx)
#bh_diff_idx = np.concatenate((lh_diff_idx, rh_diff_idx+n_hemisources), axis=0)

Reading labels from parcellation...
   read 7 labels from /Volumes/Server/NEUROLING/PersonalFiles/Nigel Flower/brainquest/mri/fsaverage/label/lh.PALS_B12_Lobes.annot
   read 10 labels from /Volumes/Server/NEUROLING/PersonalFiles/Nigel Flower/brainquest/mri/fsaverage/label/rh.PALS_B12_Lobes.annot


# Select Time Region for Analysis

In [27]:
toi_min = 0
toi_max = 800
toi_min_idx = np.squeeze(np.where(times==toi_min))
toi_max_idx = np.squeeze(np.where(times==toi_max))
toi = np.arange(toi_min, toi_max+1, 1)
toi_idx = np.arange(toi_min_idx, toi_max_idx+1, 1)

data_time = []

for i in range(n_cond):
    data_time.append(data[i][:,toi_idx,:n_sources])
    #data_list_lh.append(data_mtx[i][:,toi_idx,:n_hemisources])

In [28]:
print(data[0].shape)
print(data_time[0].shape)

(24, 901, 2562)
(24, 801, 2562)


## rmANOVA function


In [29]:
# For clustering, we don't need the p-values
return_pvals = False

def stat_fun(*args):
    # Return only the F-values
    return f_mway_rm(np.swapaxes(args, 1, 0), factor_levels=factor_levels,
                     effects=effects, return_pvals=False)[0]

## Stats Parameters

In [31]:
X = data_time
spatial_exclude = lh_diff_idx
hemi = 'lh'

tail = 1
p_thresh = 0.05
n_permutations = 10000

# Estimating F-test threshold
factor_levels = [4, 1]
effects = 'A' # A - main effect of A, B - main effect of B, A:B - interaction effect
f_thresh = f_threshold_mway_rm(n_subj, factor_levels, effects, p_thresh)

# Compute the Adjacency Matrix
src_fname = os.path.join(mri_dir, 'fsaverage', 'bem', 'fsaverage-ico-4-src.fif')
src = mne.read_source_spaces(src_fname)

adjacency = None

if hemi == 'lh':
    adjacency = mne.spatial_src_adjacency(src[:1]) # lh: src[:1], rh: src[1:]
elif hemi == 'rh':
    adjacency = mne.spatial_src_adjacency(src[1:]) # lh: src[:1], rh: src[1:]
else:
    print("ERROR: hemi must be lh or rh.")

    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    Reading a source space...
    Computing patch statistics...
    Patch information added...
    Distance information added...
    [done]
    2 source spaces read
-- number of adjacent vertices : 2562


## Spatiotemporal permutation test

In [33]:
print("Launching clustering test.")
print("Threshold: %s" % f_thresh)

F_obs, clusters, clusters_pvals, h0 = clu = \
    mne.stats.spatio_temporal_cluster_test(X, 
                                           tail = tail,
                                           threshold = f_thresh,
                                           stat_fun = stat_fun,
                                           n_permutations = n_permutations,
                                           adjacency = adjacency,
                                           spatial_exclude = spatial_exclude,
                                           out_type='indices',
                                           n_jobs = 1)

Launching clustering test.
Threshold: 2.737492307751068
stat_fun(H1): min=0.000033 max=11.726617
Running initial clustering …
Found 419 clusters


  0%|          | Permuting : 0/9999 [00:00<?,       ?it/s]

KeyboardInterrupt: 

# Save or load stats results

In [71]:
# save results
if effects == 'A':
    effect_name = 'mainQuant'
elif effects == 'B':
    effect_name = 'mainTrans'
elif effects == 'A:B':
    effect_name = 'Interaction'
    
pickle_fname = os.path.join(stats_dir, 
                            'stats_%s_%s-%s_%s.pickled' 
                            % (effect_name, str(toi_min), str(toi_max), hemi))

open_file = open(pickle_fname, "wb")
pickle.dump(clu, open_file)
open_file.close()

In [53]:
p_thresh = 0.1
print(clusters_pvals)
print(np.where(clusters_pvals < p_thresh)[0])

[0.9987 0.9914 1.     1.     1.     1.     1.     1.     1.     1.
 1.     1.     1.     1.     1.     1.     1.     1.     1.     1.
 1.     1.     1.     1.     1.     1.     1.     1.     1.     1.
 1.     1.     1.     1.     1.     1.     1.     1.     1.     1.
 1.     1.     1.     1.     1.     1.     1.     1.     1.     1.
 1.     1.     1.     1.     1.     1.     1.     1.     1.     1.
 1.     1.     1.     1.     1.     1.     1.     1.     1.     1.
 1.     1.     1.     1.     1.     1.     1.     1.     1.     1.
 1.     1.     1.     1.     1.     1.     1.     1.     1.     1.
 1.     1.     1.     1.     1.     1.     1.     0.2152 1.     1.
 1.     1.     1.     1.     1.     1.     1.     0.9996 1.     1.
 1.     1.     1.     1.     1.     1.     1.     1.     1.     1.
 1.     1.     1.     1.     1.     1.     1.     1.     1.     1.
 1.     1.     1.     1.     1.     1.     1.     1.     1.     1.
 1.     1.     1.     1.     1.     1.     1.     1.     1.   